# Week 03 — Feature Engineering & Evaluation Metrics
**ML Engineer Journey | Phase 1 — ML Fundamentals**  
**Tanggal:** 28 Mar 2026

---

## 🎯 Tujuan Notebook
Notebook ini mengintegrasikan seluruh materi Week 03 dalam satu pipeline end-to-end:

| Hari | Topik | Status |
|------|-------|--------|
| D01 | Feature Scaling & Encoding | ✅ |
| D02 | Missing Value Strategies | ✅ |
| D03 | Feature Creation & Selection | ✅ |
| D04 | Precision, Recall, F1 | ✅ |
| D05 | ROC-AUC & Threshold Tuning | ✅ |
| D06 | Improved Notebook + Polish | ← **Kamu di sini** |

---

## 📦 Dataset
Dataset yang digunakan: **Breast Cancer Wisconsin** (bawaan sklearn)  
- Binary classification: Malignant (1) vs Benign (0)  
- 30 numeric features, 569 sampel  
- Dataset dengan class imbalance ringan → cocok untuk latihan ROC vs PR curve


---
## 0. Setup & Imports

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Sklearn — data
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score

# Sklearn — preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer

# Sklearn — feature selection
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.pipeline import Pipeline

# Sklearn — model
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Sklearn — evaluation
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ All imports successful")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")

---
## 1. Load & Eksplorasi Data Awal

Sebelum melakukan apapun, pahami dulu *apa* yang ada di dataset kamu.  
Ini bukan formalitas — ini fondasi dari semua keputusan selanjutnya.

In [ ]:
# ── LOAD DATA ────────────────────────────────────────────────────────────
cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target  # 0 = malignant, 1 = benign

print(f"Shape: {df.shape}")
print(f"Target distribution:\n{df['target'].value_counts()}")
print(f"\nClass names: {cancer.target_names}")
print(f"\nMissing values: {df.isnull().sum().sum()}")

In [ ]:
# ── STATISTIK DESKRIPTIF ──────────────────────────────────────────────────
df.describe().round(2)

In [ ]:
# ── INJECT MISSING VALUES (simulasi data dunia nyata) ─────────────────────
# Di dunia nyata, data selalu ada yang kosong.
# Kita inject 5% missing secara acak untuk latihan imputation.
df_missing = df.copy()
np.random.seed(RANDOM_STATE)

n_missing = int(0.05 * df_missing.shape[0] * (df_missing.shape[1] - 1))  # 5% dari total cells
rows = np.random.randint(0, df_missing.shape[0], n_missing)
cols = np.random.randint(0, df_missing.shape[1] - 1, n_missing)  # exclude target

for r, c in zip(rows, cols):
    df_missing.iloc[r, c] = np.nan

print(f"Total missing values injected: {df_missing.isnull().sum().sum()}")
print(f"Columns dengan missing values: {(df_missing.isnull().sum() > 0).sum()}")

---
## 2. Handling Missing Values (W03D02)

**Analogi:** Missing values itu seperti lubang di peta.  
Kamu bisa (a) tempel dengan estimasi terbaik (imputation),  
atau (b) buang bagian peta yang berlubang (drop).  
Pilihan tergantung seberapa besar lubangnya dan seberapa penting area itu.

**Strategi yang dibandingkan:**
- `mean` imputation — cepat, tapi bias di distribusi skewed
- `median` imputation — lebih robust terhadap outlier
- `KNN` imputation — paling akurat, tapi paling lambat

In [ ]:
# ── MISSING VALUE IMPUTATION ──────────────────────────────────────────────
X_raw = df_missing.drop('target', axis=1).values
y = df_missing['target'].values

# Strategy 1: Mean imputation
imputer_mean = SimpleImputer(strategy='mean')
X_mean = imputer_mean.fit_transform(X_raw)

# Strategy 2: Median imputation (lebih robust untuk skewed features)
imputer_median = SimpleImputer(strategy='median')
X_median = imputer_median.fit_transform(X_raw)

# Strategy 3: KNN imputation (paling akurat, gunakan tetangga terdekat)
imputer_knn = KNNImputer(n_neighbors=5)
X_knn = imputer_knn.fit_transform(X_raw)

# Bandingkan — pakai LogReg sebagai proxy performa
scaler_temp = StandardScaler()
results_imputation = {}
for name, X_imp in [('Mean', X_mean), ('Median', X_median), ('KNN', X_knn)]:
    X_scaled = scaler_temp.fit_transform(X_imp)
    scores = cross_val_score(
        LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
        X_scaled, y, cv=5, scoring='f1'
    )
    results_imputation[name] = scores.mean()
    print(f"[{name:6}] CV F1: {scores.mean():.4f} ± {scores.std():.4f}")

# Pilih strategi terbaik
best_strategy = max(results_imputation, key=results_imputation.get)
print(f"\n✅ Strategi terpilih: {best_strategy} imputation")

# Gunakan median untuk pipeline utama (balance antara performa & kecepatan)
X_imputed = X_median.copy()

---
## 3. Feature Scaling & Encoding (W03D01)

**Analogi:** Bayangkan kamu bandingkan tinggi badan (dalam cm) dengan berat badan (dalam kg).  
Angka tinggi badan secara natural lebih besar, padahal tidak berarti lebih penting.  
Scaling menyamakan "satuan" semua fitur agar model tidak bias terhadap fitur bernilai besar.

| Scaler | Cara Kerja | Kapan Pakai |
|--------|------------|-------------|
| `StandardScaler` | mean=0, std=1 | Fitur approx normal, ada outlier ringan |
| `MinMaxScaler` | range [0,1] | Neural network, tidak ada outlier ekstrem |
| `RobustScaler` | median & IQR | Ada outlier serius |

In [ ]:
# ── FEATURE SCALING ───────────────────────────────────────────────────────
# Train-test split DULU, baru fit scaler di training set saja.
# Ini mencegah data leakage — scaler tidak boleh 'tahu' tentang test set.
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Fit HANYA di train, transform di keduanya
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled = scaler.transform(X_test)         # transform saja (NO fit)

print(f"Train set: {X_train_scaled.shape}")
print(f"Test set:  {X_test_scaled.shape}")
print(f"\nSetelah scaling — Train mean (approx 0): {X_train_scaled.mean():.4f}")
print(f"Setelah scaling — Train std  (approx 1): {X_train_scaled.std():.4f}")

---
## 4. Feature Creation & Selection (W03D03)

**Analogi:** Feature selection itu seperti packing tas untuk pendakian.  
Kamu punya 100 barang, tapi hanya bisa bawa 20.  
Bawa yang paling berguna — tinggalkan yang redundant atau tidak relevan.

Terlalu banyak fitur = model belajar noise, bukan sinyal → **overfitting**.

In [ ]:
# ── FEATURE CREATION ──────────────────────────────────────────────────────
# Buat fitur baru dari kombinasi fitur existing.
# Contoh: 'mean radius' × 'mean texture' bisa encode interaksi antara dua fitur.

feature_names = list(cancer.feature_names)

# Konversi ke DataFrame untuk kemudahan manipulasi
df_train = pd.DataFrame(X_train_scaled, columns=feature_names)
df_test  = pd.DataFrame(X_test_scaled, columns=feature_names)

# Fitur baru 1: Interaksi radius × texture
df_train['radius_x_texture'] = df_train['mean radius'] * df_train['mean texture']
df_test['radius_x_texture']  = df_test['mean radius']  * df_test['mean texture']

# Fitur baru 2: Ratio area terhadap smoothness
df_train['area_smoothness_ratio'] = df_train['mean area'] / (df_train['mean smoothness'] + 1e-8)
df_test['area_smoothness_ratio']  = df_test['mean area']  / (df_test['mean smoothness']  + 1e-8)

# Fitur baru 3: Log transform pada 'mean area' (mengatasi skewness)
df_train['log_mean_area'] = np.log1p(np.abs(df_train['mean area']))
df_test['log_mean_area']  = np.log1p(np.abs(df_test['mean area']))

print(f"Jumlah fitur sebelum creation: {len(feature_names)}")
print(f"Jumlah fitur sesudah creation: {df_train.shape[1]}")
print(f"Fitur baru: {[c for c in df_train.columns if c not in feature_names]}")

In [ ]:
# ── FEATURE SELECTION dengan SelectKBest ──────────────────────────────────
# f_classif menggunakan ANOVA F-statistic untuk mengukur relevansi fitur
# Pilih K fitur terbaik berdasarkan skor statistik

X_train_fe = df_train.values
X_test_fe  = df_test.values
feature_names_fe = list(df_train.columns)

selector = SelectKBest(score_func=f_classif, k=15)  # pilih 15 fitur terbaik
X_train_sel = selector.fit_transform(X_train_fe, y_train)  # fit di train
X_test_sel  = selector.transform(X_test_fe)                # transform test

# Tampilkan fitur terpilih
selected_mask = selector.get_support()
selected_features = [f for f, m in zip(feature_names_fe, selected_mask) if m]
feature_scores = selector.scores_[selected_mask]

print(f"Fitur terpilih ({len(selected_features)}):")
for feat, score in sorted(zip(selected_features, feature_scores), key=lambda x: -x[1]):
    print(f"  {feat:<30} F-score: {score:.2f}")

In [ ]:
# ── VISUALISASI FEATURE IMPORTANCE ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

sorted_idx = np.argsort(feature_scores)[::-1]
sorted_features = [selected_features[i] for i in sorted_idx]
sorted_scores   = [feature_scores[i] for i in sorted_idx]

colors = ['#2196F3' if i < 5 else '#90CAF9' for i in range(len(sorted_features))]
ax.barh(range(len(sorted_features)), sorted_scores[::-1], color=colors[::-1], alpha=0.85)
ax.set_yticks(range(len(sorted_features)))
ax.set_yticklabels(sorted_features[::-1], fontsize=9)
ax.set_xlabel('F-score (ANOVA)', fontsize=11)
ax.set_title('Feature Importance — SelectKBest (f_classif)', fontsize=13, fontweight='bold')
ax.axvline(x=np.mean(sorted_scores), color='red', linestyle='--', alpha=0.5, label='Mean F-score')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot disimpan sebagai feature_importance.png")

---
## 5. Model Training

Gunakan dua model untuk perbandingan evaluasi:  
1. **Logistic Regression** — baseline, interpretable, probabilistic  
2. **Random Forest** — lebih powerful, tree-based ensemble

In [ ]:
# ── TRAINING DUA MODEL ────────────────────────────────────────────────────
# Model 1: Logistic Regression
lr_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
lr_model.fit(X_train_sel, y_train)

# Model 2: Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_model.fit(X_train_sel, y_train)

print("✅ Model training selesai")
print(f"\nLogistic Regression train accuracy : {lr_model.score(X_train_sel, y_train):.4f}")
print(f"Logistic Regression test  accuracy : {lr_model.score(X_test_sel, y_test):.4f}")
print(f"\nRandom Forest train accuracy       : {rf_model.score(X_train_sel, y_train):.4f}")
print(f"Random Forest test  accuracy       : {rf_model.score(X_test_sel, y_test):.4f}")

---
## 6. Evaluasi: Precision, Recall, F1 (W03D04)

**Analogi untuk ingat bedanya:**
- **Precision:** Dari semua yang kamu *tuduh* kanker, berapa yang benar-benar kanker?
  → Penting saat *false positive* mahal (menakut-nakuti pasien sehat)
- **Recall:** Dari semua yang *benar-benar* kanker, berapa yang berhasil kamu temukan?
  → Penting saat *false negative* mahal (melewatkan pasien sakit)
- **F1:** Harmonic mean keduanya — balance ketika kedua jenis error sama pentingnya

**Dalam konteks medical screening: Recall > Precision**  
Lebih baik false alarm daripada miss a real cancer.

In [ ]:
# ── CLASSIFICATION REPORT ─────────────────────────────────────────────────
models = {'Logistic Regression': lr_model, 'Random Forest': rf_model}

for name, model in models.items():
    y_pred = model.predict(X_test_sel)
    print(f"{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

In [ ]:
# ── CONFUSION MATRIX VISUALIZATION ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_sel)
    cm = confusion_matrix(y_test, y_pred)
    
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_title(f'Confusion Matrix\n{name}', fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    
    classes = ['Malignant (0)', 'Benign (1)']
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(classes, fontsize=8)
    ax.set_yticklabels(classes, fontsize=8)
    
    # Annotasi nilai di setiap cell
    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]),
                    ha='center', va='center', fontsize=14, fontweight='bold',
                    color='white' if cm[i, j] > thresh else 'black')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. ROC-AUC & Threshold Tuning (W03D05)

### 7.1 — ROC Curve
**AUC = kemampuan ranking model**, bukan accuracy.  
AUC 0.95 artinya: jika ambil 1 pasien sakit dan 1 pasien sehat secara acak, ada 95% kemungkinan model memberi skor lebih tinggi ke pasien sakit.

**INGAT:** AUC tidak berubah saat threshold berubah — ini properti model, bukan cut-off.

In [ ]:
# ── ROC CURVE ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_model = {'Logistic Regression': '#1565C0', 'Random Forest': '#2E7D32'}

# ── SUBPLOT 1: ROC CURVE ──
ax1 = axes[0]
for name, model in models.items():
    # predict_proba()[:, 1] → probabilitas kelas positif (BUKAN predict())
    y_prob = model.predict_proba(X_test_sel)[:, 1]
    fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})',
             color=colors_model[name], lw=2)

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random Classifier (AUC = 0.5)')
ax1.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax1.set_xlabel('False Positive Rate', fontsize=11)
ax1.set_ylabel('True Positive Rate', fontsize=11)
ax1.set_title('ROC Curve', fontsize=13, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(alpha=0.3)

# ── SUBPLOT 2: PRECISION-RECALL CURVE ──
ax2 = axes[1]
for name, model in models.items():
    y_prob = model.predict_proba(X_test_sel)[:, 1]
    precision, recall, thresholds_pr = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    # GOTCHA: precision & recall panjang n+1, thresholds panjang n
    # Gunakan [:-1] saat mau plot keduanya bersama thresholds
    ax2.plot(recall, precision, label=f'{name} (AP = {ap:.3f})',
             color=colors_model[name], lw=2)

baseline_pr = y_test.mean()
ax2.axhline(y=baseline_pr, color='k', linestyle='--', alpha=0.4,
            label=f'Random baseline ({baseline_pr:.2f})')
ax2.set_xlabel('Recall', fontsize=11)
ax2.set_ylabel('Precision', fontsize=11)
ax2.set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
ax2.set_xlim([0, 1.05])
ax2.set_ylim([0, 1.05])
ax2.legend(loc='lower left')
ax2.grid(alpha=0.3)

plt.suptitle('Model Evaluation — ROC vs PR Curve', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 — Threshold Tuning

Default threshold = **0.5** — ini bukan angka ajaib, hanya konvensi.  
Untuk medical screening, kita mau **Recall tinggi** → turunkan threshold.  
**Youden's J = TPR - FPR** memberikan threshold optimal untuk balance antara sensitivity dan specificity.

In [ ]:
# ── THRESHOLD EXPERIMENT ──────────────────────────────────────────────────
# Gunakan Random Forest untuk threshold tuning
y_prob_rf = rf_model.predict_proba(X_test_sel)[:, 1]

thresholds = np.arange(0.1, 0.95, 0.05)
threshold_results = []

for t in thresholds:
    y_pred_t = (y_prob_rf >= t).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    threshold_results.append({'threshold': t, 'precision': p, 'recall': r, 'f1': f})

df_thresh = pd.DataFrame(threshold_results)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_thresh['threshold'], df_thresh['precision'], 'b-o', ms=4, label='Precision')
ax.plot(df_thresh['threshold'], df_thresh['recall'],    'r-o', ms=4, label='Recall')
ax.plot(df_thresh['threshold'], df_thresh['f1'],        'g-o', ms=4, label='F1 Score')

# Default threshold
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.6, label='Default (0.5)')

# Youden's J threshold
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob_rf)
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
best_threshold = roc_thresholds[best_idx]
ax.axvline(x=best_threshold, color='purple', linestyle='-.',
           alpha=0.8, label=f"Youden's J ({best_threshold:.2f})")

ax.set_xlabel('Threshold', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Threshold Tuning — Precision / Recall / F1 Tradeoff\n(Random Forest)', 
             fontsize=13, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim([0.05, 1.0])

plt.tight_layout()
plt.savefig('threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📌 Youden's J optimal threshold: {best_threshold:.3f}")
print(f"   (TPR={tpr[best_idx]:.3f}, FPR={fpr[best_idx]:.3f})")

In [ ]:
# ── PERBANDINGAN: DEFAULT vs OPTIMAL THRESHOLD ────────────────────────────
print("=" * 55)
print("  Threshold Comparison — Random Forest")
print("=" * 55)
print(f"{'Metric':<15} {'Default (0.50)':>15} {'Optimal ({:.2f})'.format(best_threshold):>16}")
print("-" * 55)

for t_val, t_name in [(0.5, 'Default'), (best_threshold, 'Optimal')]:
    pass  # values computed below

y_pred_default = (y_prob_rf >= 0.50).astype(int)
y_pred_optimal = (y_prob_rf >= best_threshold).astype(int)

for metric, fn in [('Precision', precision_score), ('Recall', recall_score), ('F1', f1_score)]:
    v_def = fn(y_test, y_pred_default, zero_division=0)
    v_opt = fn(y_test, y_pred_optimal, zero_division=0)
    delta = v_opt - v_def
    print(f"{metric:<15} {v_def:>15.4f} {v_opt:>16.4f}  {'↑' if delta > 0 else '↓'}{abs(delta):.4f}")

print("=" * 55)

---
## 8. Rangkuman Pipeline

Ini adalah gambaran keseluruhan pipeline yang sudah dibangun di notebook ini:

In [ ]:
# ── FINAL SUMMARY TABLE ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  WEEK 03 — PIPELINE SUMMARY")
print("=" * 60)

summary = [
    ("Dataset",         "Breast Cancer Wisconsin (569 samples, 30 features)"),
    ("Missing Values",  "5% injected → Median Imputation"),
    ("Scaling",         "StandardScaler (fit on train only)"),
    ("Feature Eng.",    "3 new features created (interaction, ratio, log)"),
    ("Feature Sel.",    "SelectKBest (f_classif, k=15) → 15 features"),
    ("Models",          "Logistic Regression + Random Forest"),
    ("Best AUC",        f"{max(auc(roc_curve(y_test, m.predict_proba(X_test_sel)[:,1])[0], roc_curve(y_test, m.predict_proba(X_test_sel)[:,1])[1]) for m in models.values()):.4f} (Random Forest)"),
    ("Optimal Threshold", f"{best_threshold:.3f} (Youden's J)"),
]

for key, val in summary:
    print(f"  {key:<20}: {val}")

print("=" * 60)
print("\n✅ Notebook Week 03 selesai. Siap push ke GitHub.")

---
## 📌 Key Takeaways Week 03

1. **Scaling wajib fit di train set saja** — kalau fit di full data, kamu bocorkan informasi test ke model (data leakage)
2. **Imputation strategy tergantung distribusi** — median lebih robust dari mean untuk skewed data
3. **Feature creation bukan asal tambah** — tambah fitur yang punya makna domain
4. **Precision vs Recall tradeoff** bisa di-control dengan threshold — default 0.5 bukan angka sakral
5. **AUC tidak berubah saat threshold berubah** — ini properti intrinsik model
6. **ROC curve optimistis di imbalanced data** — selalu cek PR curve juga
7. **Youden's J = argmax(TPR - FPR)** — threshold optimal untuk medical screening

---
*Week 03 — Feature Engineering & Evaluation Metrics | ML Engineer Journey*  
*github.com/lolivampire/ML-Project*